## ConvoKit Conversion

**Dataset:** Unintended Offense Dataset

**Paper:** C. W. Tsai, Y.-H. Huang, T.-K. Liao, D. F. S. Estrada, R. Latifah, and Y.-S. Chen, “Leveraging Conflicts in Social Media Posts: Unintended Offense Dataset,” in Proceedings of the 2024 Conference on Empirical Methods in Natural Language Processing, Y. Al-Onaizan, M. Bansal, and Y.-N. Chen, Eds., Miami, Florida, USA: Association for Computational Linguistics, Nov. 2024, pp. 4512–4522. Accessed: Nov. 16, 2024. [Online]. Available: https://aclanthology.org/2024.emnlp-main.259

**Dataset:** https://github.com/IDEA-NTHU-Taiwan/unintended-offense-tweets/tree/main

In [ ]:
import json
from convokit import Corpus, Utterance, Speaker, Conversation
from datetime import datetime
import pandas as pd

In [ ]:
uo_dataset = r'<path>/conversations_text_attr_offense.json' #created by mergeJSON.py and mergeJSON_CSV.py. Merged dataset located in ./merged-data

In [ ]:
with open(uo_dataset, "r", encoding="utf-8") as f:
    data = json.load(f)

Speaker objects

In [ ]:
speakers = {}
def create_speaker(speaker_id, **meta): #pass in speaker id and any metadata
    if speaker_id not in speakers:
        speakers[speaker_id] = Speaker(id=speaker_id, meta=dict(meta))
    else:
        speakers[speaker_id].meta.update(meta)
    return speakers[speaker_id]

Utterances (Each utterance is a single Tweet, so this includes Context Tweets, Target Tweets, Follow Up Tweets, and Cue Tweets)

In [ ]:
utterances = []
convo_checks = {}

for convo in data:
    convo_id = convo["conversation_id"] #set convo id

    utterance_idx = 0 #for each convo, start index at 0

    #Context Tweets
    last_context = None #no context Tweets before root context Tweet
    for tweet in convo.get("context_tweets", []):
        metadata = tweet.get("_meta", {}) or {} #get metadata from _meta object in JSON
        metrics  = metadata.get("public_metrics", {}) or {} #and from the nested public_metrics object

        unique_id = f"{convo_id}_{utterance_idx}" #strcuture of utterance id
        create_speaker(tweet["author_id"]) #call create_speaker() method to create speaker object from author id

        utterances.append(Utterance( #create context Tweet utterance in convokit format (6 required fields plus metadata)
            id = unique_id,
            speaker = speakers[tweet["author_id"]],
            conversation_id = convo_id,
            reply_to = last_context,
            text = tweet["text"],
            timestamp = ( #Unix timestamp
                int(datetime.fromisoformat(metadata.get("created_at").replace("Z","+00:00")).timestamp())
                if metadata.get("created_at") else None
                ),
            meta = {
                "tweet_type": "context",
                "index": utterance_idx,
                "device": metadata.get("source"),
                "reply_settings": metadata.get("reply_settings"),
                "retweet_count": metrics.get("retweet_count", 0),
                "reply_count": metrics.get("reply_count", 0),
                "like_count": metrics.get("like_count", 0),
                "quote_count": metrics.get("quote_count", 0),
            }
        ))
        last_context = unique_id #set last context to this context
        utterance_idx += 1 #increment index

    #Target Tweets
    target_ids = [] #list of target IDs, in case corpus includes more than one per convo
    for tweet in convo.get("target_tweet", []):
        metadata = tweet.get("_meta", {}) or {} #get metadata from _meta object in JSON
        metrics  = metadata.get("public_metrics", {}) or {} #and from the nested public_metrics object

        unique_id = f"{convo_id}_{utterance_idx}" #strcuture of utterance id
        create_speaker(tweet["author_id"]) #call create_speaker() method to create speaker object from author id

        meta = { #initiatlize meta earlier to check to see if the target Tweet has an associated offensiveness and confidence score
            "tweet_type": "target",
            "index": utterance_idx,
            "device": metadata.get("source"),
            "reply_settings": metadata.get("reply_settings"),
            "retweet_count": metrics.get("retweet_count", 0),
            "reply_count": metrics.get("reply_count", 0),
            "like_count": metrics.get("like_count", 0),
            "quote_count": metrics.get("quote_count", 0),
        }
        #if offensiveness and confidence scores exist, include in metadata
        if "offensiveness" in tweet: meta["offensiveness"] = tweet["offensiveness"]
        if "confidence" in tweet:  meta["confidence"] = tweet["confidence"]

        utterances.append(Utterance( #create target Tweet utterance in convokit format
            id = unique_id,
            speaker = speakers[tweet["author_id"]],
            conversation_id = convo_id,
            reply_to = last_context,
            text = tweet["text"],
            timestamp=( #Unix timestamp
                int(datetime.fromisoformat(metadata.get("created_at").replace("Z","+00:00")).timestamp())
                if metadata.get("created_at") else None
                ),
            meta = meta #set to meta from above
        ))
        target_ids.append(unique_id) #if more than one target, append to target list
        utterance_idx += 1 #increment index
    last_target = target_ids[-1] if target_ids else None #set last target to last target in list
    #I don't think there are more than one, but just adding as a fail safe

    #Follow Up Tweets
    follow_ids = [] #create follow up list (same fail safe rationale as above)
    for tweet in convo.get("follow-up_tweet", []):
        metadata = tweet.get("_meta", {}) or {} #get metadata from _meta object in JSON
        metrics = metadata.get("public_metrics", {}) or {} #and from the nested public_metrics object

        unique_id = f"{convo_id}_{utterance_idx}" #strcuture of utterance id
        create_speaker(tweet["author_id"]) #call create_speaker() method to create speaker object from author id

        utterances.append(Utterance( #create follow-up Tweet utterance in convokit format (6 required fields plus metadata)
            id = unique_id,
            speaker = speakers[tweet["author_id"]],
            conversation_id = convo_id,
            reply_to = last_target,
            text = tweet["text"],
            timestamp = ( #Unix timestamp
                int(datetime.fromisoformat(metadata.get("created_at").replace("Z","+00:00")).timestamp())
                if metadata.get("created_at") else None
                ),
            meta = {
                "tweet_type": "followup",
                "index": utterance_idx,
                "device": metadata.get("source"),
                "reply_settings": metadata.get("reply_settings"),
                "retweet_count": metrics.get("retweet_count", 0),
                "reply_count": metrics.get("reply_count", 0),
                "like_count": metrics.get("like_count", 0),
                "quote_count": metrics.get("quote_count", 0),
            }
        ))
        follow_ids.append(unique_id) #if more than one follow up, append to follow up list
        utterance_idx += 1 #increment index
    last_follow = follow_ids[-1] if follow_ids else None #set last target to last target in list

    #Cue Tweets
    for tweet in convo.get("cue_tweets", []):
        metadata = tweet.get("_meta", {}) or {} #get metadata from _meta object in JSON
        metrics = metadata.get("public_metrics", {}) or {} #and from the nested public_metrics object

        unique_id = f"{convo_id}_{utterance_idx}" #strcuture of utterance id
        create_speaker(tweet["author_id"]) #call create_speaker() method to create speaker object from author id

        utterances.append(Utterance( #create cue Tweet utterance in convokit format (6 required fields plus metadata)
            id = unique_id,
            speaker = speakers[tweet["author_id"]],
            conversation_id = convo_id,
            reply_to = last_follow,
            text = tweet["text"],
            timestamp = ( #Unix timestamp
                int(datetime.fromisoformat(metadata.get("created_at").replace("Z","+00:00")).timestamp())
                if metadata.get("created_at") else None
                ),
            meta={
                "tweet_type": "cue",
                "index": utterance_idx,
                "device": metadata.get("source"),
                "reply_settings": metadata.get("reply_settings"),
                "retweet_count": metrics.get("retweet_count", 0),
                "reply_count": metrics.get("reply_count", 0),
                "like_count": metrics.get("like_count", 0),
                "quote_count": metrics.get("quote_count", 0),
            }
        ))
        utterance_idx += 1 #increment index

Create corpus

In [ ]:
corpus = Corpus(utterances=utterances)

Basic checks

In [ ]:
print("utterances:", len(list(corpus.iter_utterances())))
print("speakers:", len(list(corpus.iter_speakers())))
print("conversations:", len(list(corpus.iter_conversations())))

In [ ]:
utterances_df = pd.DataFrame([u.to_dict() for u in corpus.iter_utterances()])
utterances_df.head(10)

In [ ]:
print(utterances_df.loc[1, 'meta'])

Dump Corpus

In [ ]:
OUT_DIR = "<path>/unintended_offense_corpus"
corpus.dump(OUT_DIR)